# 03 — XGBoost: class-weight vs SMOTE, plus a `loan_grade` ablation

Three models trained and compared, each with a documented reason for existing:

1. **XGBoost + `scale_pos_weight`** — no synthetic data, the recommended starting
   point per the project plan.
2. **XGBoost + SMOTE** (fit on train only, after the split — never before) — compared
   honestly against (1), not applied blindly.
3. **XGBoost ablation, without `loan_grade`/`loan_int_rate`** — EDA (notebook 01)
   found `loan_grade` alone nearly separates the classes (9.96% → 98.4% default rate
   across grades A→G). This model asks: how well can applicant-level attributes
   alone predict risk, without the lender's own risk rating handed to it? A more
   defensible story than one AUC number from a model that leans entirely on grade.

In [1]:
import sys, json, os
sys.path.append('..')

import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

from src.preprocessing import TARGET, CATEGORICAL_COLS, NUMERIC_COLS
from src.metrics import summarize

pd.set_option('display.width', 120)
N_JOBS = -1  # use all available cores (10 on this machine) per XGBoost fit

In [2]:
train = pd.read_csv('../data/processed/train.csv')
val = pd.read_csv('../data/processed/val.csv')
test = pd.read_csv('../data/processed/test.csv')

X_train, y_train = train.drop(columns=[TARGET]), train[TARGET]
X_val, y_val = val.drop(columns=[TARGET]), val[TARGET]
X_test, y_test = test.drop(columns=[TARGET]), test[TARGET]

ratio = (y_train == 0).sum() / (y_train == 1).sum()
print('scale_pos_weight (train class ratio):', round(ratio, 4))

scale_pos_weight (train class ratio): 3.573


## Shared encoding (one-hot; low cardinality — max 7 levels — so no column-explosion
risk here, unlike Home Credit's 58-level `ORGANIZATION_TYPE`). Fit on train only.

In [3]:
encoder = ColumnTransformer([
    ('num', 'passthrough', NUMERIC_COLS),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False), CATEGORICAL_COLS),
], verbose_feature_names_out=False)
encoder.set_output(transform='pandas')

X_train_enc = encoder.fit_transform(X_train)
X_val_enc = encoder.transform(X_val)
X_test_enc = encoder.transform(X_test)
print('encoded feature count:', X_train_enc.shape[1])
print(list(X_train_enc.columns))

encoded feature count: 22
['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'person_home_ownership_OTHER', 'person_home_ownership_OWN', 'person_home_ownership_RENT', 'loan_intent_EDUCATION', 'loan_intent_HOMEIMPROVEMENT', 'loan_intent_MEDICAL', 'loan_intent_PERSONAL', 'loan_intent_VENTURE', 'loan_grade_B', 'loan_grade_C', 'loan_grade_D', 'loan_grade_E', 'loan_grade_F', 'loan_grade_G', 'cb_person_default_on_file_Y']


## Model 1 — XGBoost + `scale_pos_weight`

In [4]:
xgb_weighted = XGBClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=5,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=ratio, eval_metric='auc',
    early_stopping_rounds=50, random_state=42, n_jobs=N_JOBS,
)
xgb_weighted.fit(X_train_enc, y_train, eval_set=[(X_val_enc, y_val)], verbose=False)

proba_weighted_val = xgb_weighted.predict_proba(X_val_enc)[:, 1]
proba_weighted_test = xgb_weighted.predict_proba(X_test_enc)[:, 1]
print('best_iteration:', xgb_weighted.best_iteration)
print('VAL :', {k: round(v, 4) for k, v in summarize(y_val, proba_weighted_val).items()})
print('TEST:', {k: round(v, 4) for k, v in summarize(y_test, proba_weighted_test).items()})

best_iteration: 436
VAL : {'auc': 0.9471, 'ks': 0.7556, 'gini': 0.8942, 'pr_auc': 0.9034, 'brier': 0.0703}
TEST: {'auc': 0.9459, 'ks': 0.7556, 'gini': 0.8919, 'pr_auc': 0.9025, 'brier': 0.0681}


## Model 2 — XGBoost + SMOTE

SMOTE fit on the **encoded training matrix only**, strictly after the split. Val/test
remain untouched real data — this is the leakage rule from the project plan, applied,
not just stated.

In [5]:
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train_enc, y_train)
print('before SMOTE:', X_train_enc.shape, dict(y_train.value_counts()))
print('after  SMOTE:', X_train_res.shape, dict(y_train_res.value_counts()))

xgb_smote = XGBClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=5,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='auc', early_stopping_rounds=50, random_state=42, n_jobs=N_JOBS,
)
xgb_smote.fit(X_train_res, y_train_res, eval_set=[(X_val_enc, y_val)], verbose=False)

proba_smote_val = xgb_smote.predict_proba(X_val_enc)[:, 1]
proba_smote_test = xgb_smote.predict_proba(X_test_enc)[:, 1]
print('best_iteration:', xgb_smote.best_iteration)
print('VAL :', {k: round(v, 4) for k, v in summarize(y_val, proba_smote_val).items()})
print('TEST:', {k: round(v, 4) for k, v in summarize(y_test, proba_smote_test).items()})

before SMOTE: (19449, 22) {0: np.int64(15196), 1: np.int64(4253)}
after  SMOTE: (30392, 22) {1: np.int64(15196), 0: np.int64(15196)}


best_iteration: 496
VAL : {'auc': 0.9458, 'ks': 0.7524, 'gini': 0.8917, 'pr_auc': 0.9023, 'brier': 0.0531}
TEST: {'auc': 0.9439, 'ks': 0.756, 'gini': 0.8877, 'pr_auc': 0.9013, 'brier': 0.0534}


## Class-weight vs SMOTE — honest comparison

In [6]:
comparison = pd.DataFrame({
    'scale_pos_weight': summarize(y_test, proba_weighted_test),
    'smote': summarize(y_test, proba_smote_test),
    'logistic_regression_baseline': json.load(open('../reports/baseline_results.json'))['test'],
}).T
print(comparison.round(4))

                                 auc      ks    gini  pr_auc   brier
scale_pos_weight              0.9459  0.7556  0.8919  0.9025  0.0681
smote                         0.9439  0.7560  0.8877  0.9013  0.0534
logistic_regression_baseline  0.8697  0.6038  0.7394  0.7166  0.1384


## Model 3 — Ablation: no `loan_grade` / `loan_int_rate`

Same class-weighted XGBoost setup, with the lender's own risk rating and its direct
correlate removed, to see how much signal survives in applicant-level attributes
alone.

In [7]:
drop_cols = [c for c in X_train_enc.columns if c.startswith('loan_grade') or c == 'loan_int_rate']
print('dropping:', drop_cols)

X_train_abl = X_train_enc.drop(columns=drop_cols)
X_val_abl = X_val_enc.drop(columns=drop_cols)
X_test_abl = X_test_enc.drop(columns=drop_cols)

xgb_ablation = XGBClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=5,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=ratio, eval_metric='auc',
    early_stopping_rounds=50, random_state=42, n_jobs=N_JOBS,
)
xgb_ablation.fit(X_train_abl, y_train, eval_set=[(X_val_abl, y_val)], verbose=False)

proba_ablation_test = xgb_ablation.predict_proba(X_test_abl)[:, 1]
print('best_iteration:', xgb_ablation.best_iteration)
print('TEST (no grade/int_rate):', {k: round(v, 4) for k, v in summarize(y_test, proba_ablation_test).items()})
print('TEST (full model)       :', {k: round(v, 4) for k, v in summarize(y_test, proba_weighted_test).items()})

dropping: ['loan_int_rate', 'loan_grade_B', 'loan_grade_C', 'loan_grade_D', 'loan_grade_E', 'loan_grade_F', 'loan_grade_G']


best_iteration: 496
TEST (no grade/int_rate): {'auc': 0.9091, 'ks': 0.6568, 'gini': 0.8181, 'pr_auc': 0.8286, 'brier': 0.1011}
TEST (full model)       : {'auc': 0.9459, 'ks': 0.7556, 'gini': 0.8919, 'pr_auc': 0.9025, 'brier': 0.0681}


In [8]:
os.makedirs('../reports', exist_ok=True)
results = {
    'scale_pos_weight_ratio': float(ratio),
    'xgb_scale_pos_weight': {'val': summarize(y_val, proba_weighted_val), 'test': summarize(y_test, proba_weighted_test), 'best_iteration': int(xgb_weighted.best_iteration)},
    'xgb_smote': {'val': summarize(y_val, proba_smote_val), 'test': summarize(y_test, proba_smote_test), 'best_iteration': int(xgb_smote.best_iteration)},
    'xgb_ablation_no_grade': {'test': summarize(y_test, proba_ablation_test), 'best_iteration': int(xgb_ablation.best_iteration), 'dropped_columns': drop_cols},
}
with open('../reports/xgboost_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=float)
results

{'scale_pos_weight_ratio': 3.57300728897249,
 'xgb_scale_pos_weight': {'val': {'auc': 0.9471202992967306,
   'ks': 0.7556348443031311,
   'gini': 0.8942405985934612,
   'pr_auc': 0.9034371165951286,
   'brier': 0.07034740110195756},
  'test': {'auc': 0.9459323112628397,
   'ks': 0.7556213140487671,
   'gini': 0.8918646225256794,
   'pr_auc': 0.9025023551626242,
   'brier': 0.06809926408325692},
  'best_iteration': 436},
 'xgb_smote': {'val': {'auc': 0.9458333623403513,
   'ks': 0.7523629665374756,
   'gini': 0.8916667246807026,
   'pr_auc': 0.9023375239693762,
   'brier': 0.053069875851970164},
  'test': {'auc': 0.9438605053630581,
   'ks': 0.7559603452682495,
   'gini': 0.8877210107261162,
   'pr_auc': 0.9012545823071781,
   'brier': 0.05340387337467289},
  'best_iteration': 496},
 'xgb_ablation_no_grade': {'test': {'auc': 0.909058954940066,
   'ks': 0.6568127274513245,
   'gini': 0.8181179098801321,
   'pr_auc': 0.8286252289964224,
   'brier': 0.10106053477168325},
  'best_iteration'

## Summary

- Encoded features (one-hot, low cardinality, no explosion risk): see feature list
  above.
- `scale_pos_weight` vs SMOTE compared honestly on identical test data — whichever
  wins (or ties) is used going forward; SMOTE is not applied by default just because
  it's a known technique.
- Ablation model confirms how much predictive power survives without `loan_grade`/
  `loan_int_rate` — the defensible story for feature importance discussed in EDA.
- All results saved to `reports/xgboost_results.json` for the calibration and SHAP
  notebooks to reference.